# 2.3 Join and Bind

**Core series, class 5.** Our data is in four tables: the cohort, the measurements, the survey answers, and the CKD dates. An analysis needs them **together**, one row per person. **Binding** stacks tables on top of each other; **joining** lines them up side by side by a shared **key** (`person_id`). Joins are where analyses most often go silently wrong, so we learn the join types with a small example first, and **check** every join.

### By the end of this notebook you'll be able to

1. Stack tables with `bind_rows()`, and say when that's the right tool
2. Explain **left**, **inner**, **full**, and **anti** joins, and choose the right one
3. Check a join: **row counts** before and after, **unique keys**, **unmatched IDs**, and **key types**
4. Build the analysis table that 3.1–3.4 use

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | How do we combine four tables into one, without losing or duplicating people? |
| 🧪 **The analysis** | `bind_rows()` to stack; `left_join()` to combine by `person_id`; `anti_join()` and counts to check |
| 🎯 **Why** | A wrong join can silently **drop** people (inner instead of left) or **duplicate** them (a key that isn't unique), changing every result after it |
| ✅ **What we conclude** | One analysis table: 10,000 rows, one per person, with every variable |
| ⚠️ **What it can't tell us** | Anything about diabetes yet: that's 3.1–3.4 |
| 🔗 **Connections** | Uses the files from **1.2**, **2.1**, and **2.2**; saves `final_analysis.csv` for **3.1–3.4** |

| | |
|---|---|
| 📂 **Uses** | `cohort_clean.csv`, `measurements_wide.csv`, `survey_wide.csv`, `ckd_dates.csv` |
| 💾 **Saves** | `final_analysis.csv` |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts: bind and join | 15 |
| 2. Load | 5 |
| 3. Binding rows | 10 |
| 4. The join types, on a tiny example | 15 |
| 5. Checks before joining | 10 |
| 6. Build the analysis table | 15 |
| Wrap-up | 5 |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 75 minutes |
| ⏱️ **Run time** | under **1 minute** to run every cell |
| 💰 **Data cost** | **$0**: no BigQuery queries |
| 💰 **App cost** | about **$0.27 an hour** while the app is running. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| **Estimated total** | about **$0.35** per student for this class (app time only) |

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts: bind and join

| | **Bind** | **Join** |
|---|---|---|
| What it does | stacks tables **on top of** each other | lines tables up **side by side** |
| The tables have | the **same columns**, different rows (people) | **different columns**, the same people |
| Matched by | column **names** | a **key** column, e.g. `person_id` |
| Function | `bind_rows()` | `left_join()`, `inner_join()`, `full_join()`, `anti_join()` |
| Example | cases + controls → everyone | everyone + their measurements |

### The join types

Two tables, **x** (on the left) and **y**, matched by a key:

```
     x            y
  ┌─────┐      ┌─────┐
  │  A  │      │  A  │      A, B: in both
  │  B  │      │  B  │      C: only in x
  │  C  │      │  D  │      D: only in y
  └─────┘      └─────┘
```

| Join | Keeps | Result | Use it when |
|---|---|---|---|
| `left_join(x, y)` | **every row of x**; y's columns where they match, `NA` where they don't | A, B, C | adding information to a main table **without losing anyone**: our default |
| `inner_join(x, y)` | only rows in **both** | A, B | you only want people with both kinds of data (and know who you're dropping) |
| `full_join(x, y)` | rows in **either** | A, B, C, D | combining two lists of people |
| `anti_join(x, y)` | rows of x with **no** match in y | C | **checking**: who doesn't match? |

### Keys

The **key** must mean the same thing, and have the same **type**, in both tables. Text `"1234567"` and the number `1234567` don't match. If the key appears **more than once** in y (e.g. many measurements per person), a join **repeats** each x row once per match: **one-to-many**. That's sometimes what you want, but usually a sign you forgot to reshape first (2.2).

## Part 2. Load

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

cohort_clean      <- load_data("cohort_clean.csv",      col_types = cols(person_id = "c"))
measurements_wide <- load_data("measurements_wide.csv", col_types = cols(person_id = "c"))
survey_wide       <- load_data("survey_wide.csv",       col_types = cols(person_id = "c"))
ckd_dates         <- load_data("ckd_dates.csv",         col_types = cols(person_id = "c"))

c(cohort = nrow(cohort_clean), measurements = nrow(measurements_wide),
  survey = nrow(survey_wide), ckd = nrow(ckd_dates))

**🔍 Read the output:** 10,000 people in the cohort; fewer in each of the other tables, because not everyone has measurements, survey answers, or CKD.

## Part 3. Binding rows

Imagine the cases and controls had been downloaded **separately** (as in older versions of this course). `bind_rows()` stacks them back into one table. `.id = "source"` adds a column saying which table each row came from.

In [ ]:
cases    <- cohort_clean %>% filter(t2d == "Yes")
controls <- cohort_clean %>% filter(t2d == "No")

everyone <- bind_rows(case = cases, control = controls, .id = "source")

c(cases = nrow(cases), controls = nrow(controls), together = nrow(everyone))
everyone %>% count(source, t2d)

**🔍 Read the output:** the rows add up (5,000 + 5,000 = 10,000), and `source` matches `t2d`.

`bind_rows()` matches columns by **name**. If one table had an extra column, the other table's rows would get `NA` in it; if a column had a different **name** in each table, you'd get two half-empty columns. Check `names()` before binding tables from different sources.

**When to bind:** the tables describe **different people** (or different times) with the **same variables**.

## Part 4. The join types, on a tiny example

Joins are easiest to understand on a table small enough to see. These are **made-up** people, not *All of Us* data:

In [ ]:
people <- tibble(person_id = c("A", "B", "C"), age = c(45, 60, 52))
labs   <- tibble(person_id = c("A", "B", "D"), BMI = c(31, 24, 28))

left_join(people, labs, by = "person_id")    # every person; BMI where known
inner_join(people, labs, by = "person_id")   # only people in both
full_join(people, labs, by = "person_id")    # everyone in either
anti_join(people, labs, by = "person_id")    # people with no BMI

**🔍 Read the output:**

- `left_join`: A, B, C. C has `BMI = NA`: no lab result, but still in the data.
- `inner_join`: only A and B. **C disappeared**, silently.
- `full_join`: A, B, C, D. D has no age: they weren't in `people`.
- `anti_join`: just C: the person who would be lost by an inner join.

`by = "person_id"` names the key. Always write it, even when R could guess: it documents the join.

### ✍️ Your turn

Add a **second** BMI row for person A to `labs` (as if A had two results), then `left_join()` again. How many rows do you get, and why?

<details>
<summary><b>Show answer</b></summary>

```r
labs2 <- bind_rows(labs, tibble(person_id = "A", BMI = 33))
left_join(people, labs2, by = "person_id")
```

**4 rows**: A appears **twice**, once per BMI. The key wasn't unique in `labs2`, so the join became **one-to-many**. On real data this silently multiplies people, and every count and mean after it is wrong. That's why we reshape to one row per person (2.2) **before** joining, and check the row count **after**.

</details>

In [ ]:
# Your code here

## Part 5. Checks before joining

Three checks, every time:

1. **Same key type** in both tables (all ours are text)
2. The key is **unique** in the table being added (one row per person)
3. **Who won't match?** (`anti_join()`), in both directions

In [ ]:
# 1. Key types
sapply(list(cohort = cohort_clean, measurements = measurements_wide, survey = survey_wide, ckd = ckd_dates),
       function(df) class(df$person_id))

# 2. Unique keys: the number of duplicated IDs in each table (should all be 0)
sapply(list(cohort = cohort_clean, measurements = measurements_wide, survey = survey_wide, ckd = ckd_dates),
       function(df) sum(duplicated(df$person_id)))

# 3. Unmatched IDs
c(cohort_without_measurements = nrow(anti_join(cohort_clean, measurements_wide, by = "person_id")),
  measurements_not_in_cohort  = nrow(anti_join(measurements_wide, cohort_clean, by = "person_id")))

**🔍 Read the output:**

1. Every table's `person_id` is `character`: they'll match.
2. All zeros: one row per person in every table, so no join will duplicate anyone.
3. Some people in the cohort have **no** measurements: they'll get `NA`s from a left join (an inner join would drop them). **No** measurements belong to people outside the cohort: if that number weren't 0, something would be wrong with the IDs.

### What a type mismatch looks like

If one table's IDs were numbers, the join fails with an error (which is better than silently matching nothing):

In [ ]:
ids_as_numbers <- measurements_wide %>% mutate(person_id = as.numeric(person_id))

tryCatch(left_join(cohort_clean, ids_as_numbers, by = "person_id"),
         error = function(e) conditionMessage(e))

**🔍 Read the output:** *Can't join … due to incompatible types (`<character>` and `<double>`)*. Fix it by giving both keys the same type, ideally when reading the files (`col_types = cols(person_id = "c")`). `tryCatch()` lets the notebook show the error message instead of stopping.

## Part 6. Build the analysis table

Start from the **cohort** (everyone), and `left_join()` each table onto it, so no one is lost. Then add an **age group** for tables. `cut()` turns a number into groups: `right = FALSE` means each group includes its lower bound (18 is in "18-39").

In [ ]:
final_analysis <- cohort_clean %>%
  left_join(measurements_wide, by = "person_id") %>%
  left_join(survey_wide,       by = "person_id") %>%
  left_join(ckd_dates,         by = "person_id") %>%
  mutate(age_group = cut(age, breaks = c(18, 40, 65, Inf), right = FALSE,
                         labels = c("18-39", "40-64", "65+")))

# The check that matters most: still one row per person?
c(rows = nrow(final_analysis), people = n_distinct(final_analysis$person_id))

# How complete is each variable?
final_analysis %>%
  summarise(across(c(BMI, SBP, HbA1c, LDL, ptsd_doctor, ptsd_treatment, first_ckd_date),
                   ~ round(100 * mean(!is.na(.x)), 1)))

save_data(final_analysis, "final_analysis.csv")

**🔍 Read the output:**

- **10,000 rows, 10,000 people**: no one lost, no one duplicated.
- The percent of people with each variable: nearly everyone has BMI and SBP; fewer have HbA1c and LDL; only a few percent have PTSD answers; `first_ckd_date` is filled in only for people with CKD.

We kept every `NA`: 3.1–3.4 each use the people who have **their** variables.

**💬 Discuss:** compared with the old version of this course, which used `drop_na()` here, how many people would that have left? Why would those people be different from everyone else? (Check with `nrow(drop_na(final_analysis, BMI, SBP, HbA1c, LDL, ptsd_doctor, ptsd_treatment))`.)

## ⚠️ Common errors

| Message | Means | Fix |
|---|---|---|
| *Can't join … due to incompatible types* | Key types differ | `col_types = cols(person_id = "c")` when reading, or `as.character()` |
| *Detected an unexpected many-to-many relationship* | The key isn't unique in **both** tables | Reshape to one row per person first (2.2), or check for duplicates |
| More rows after a join than before | One-to-many: the key isn't unique in y | `sum(duplicated(y$person_id))` |
| Fewer rows after a join | You used `inner_join()` | Use `left_join()` from the main table |
| A column appears as `x.1` / `.x` and `.y` | Both tables have a column with the same name | Drop or rename one before joining |

## ✅ Key takeaways

- **Bind** stacks tables with the same columns; **join** combines tables by a key
- `left_join()` from your main table keeps **everyone**; `inner_join()` silently drops people
- **Check** every join: key types, unique keys, `anti_join()`, and row counts after
- Keep `NA`s; decide per analysis

**Next:** **3.1 table_one**: describe cases and controls.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).